# Random Forest EcoPlant — latih dengan gambar sendiri

Struktur folder yang diharapkan: satu folder per kelas, isi gambarnya bebas (JPEG/PNG).

```
folder_gambar/
├── cordana/
├── healthy/
├── pestalotiopsis/
└── sigatoka/
```

Nama folder menjadi label. Notebook ini memakai split `train/val/test` bila ketiganya ada;
kalau tidak, gambar dibagi sendiri 70/15/15 secara stratified (seed 42).

Aplikasi EcoPlant mengunci 4 kelas: Cordana, Healthy, Pestalotiopsis, Sigatoka. Model dengan
kelas berbeda tetap bisa dilatih di sini, tetapi akan ditolak aplikasi saat validasi label.

## Memakai gambar dari HP atau Google Colab

Di Colab, unggah zip berisi folder kelas lalu ekstrak:

```python
from google.colab import files
files.upload()          # pilih gambar.zip
!unzip -q gambar.zip -d /content/gambar
```

Kemudian isi `DATA_DIR_OVERRIDE = "/content/gambar"` pada sel berikut.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

SEED = 42
IMAGE_SIZE = (64, 64)
BINS = 16
N_TREES = 200
APP_CLASSES = ["Cordana", "Healthy", "Pestalotiopsis", "Sigatoka"]

DATA_DIR_OVERRIDE = ""      # contoh: "/content/gambar" atau "D:/dataset/daun"
OUTPUT_DIR_OVERRIDE = ""    # kosong = training/artifacts_rf_custom
COPY_TO_ASSETS = True       # salin forest.json ke ecoplant/assets/models

In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ROOT = Path.cwd()
if not (ROOT / "training").is_dir() and (ROOT.parent / "training").is_dir():
    ROOT = ROOT.parent

if DATA_DIR_OVERRIDE.strip():
    DATA_DIR = Path(DATA_DIR_OVERRIDE).expanduser()
elif (ROOT / "training" / "dataset_retrain").is_dir():
    DATA_DIR = ROOT / "training" / "dataset_retrain"
else:
    raise FileNotFoundError("Folder gambar tidak ditemukan. Isi DATA_DIR_OVERRIDE.")

OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser()
    if OUTPUT_DIR_OVERRIDE.strip()
    else ROOT / "training" / "artifacts_rf_custom"
)
print("Dataset:", DATA_DIR.resolve())
print("Output :", OUTPUT_DIR.resolve())

In [ ]:
def collect(folder):
    return [
        {"path": path, "label": path.parent.name.capitalize()}
        for path in sorted(folder.rglob("*"))
        if path.is_file() and path.suffix.lower() in EXTENSIONS
    ]

if all((DATA_DIR / name).is_dir() for name in ("train", "val", "test")):
    parts = {name: pd.DataFrame(collect(DATA_DIR / name)) for name in ("train", "val", "test")}
    print("Memakai split yang sudah ada di", DATA_DIR)
else:
    frame = pd.DataFrame(collect(DATA_DIR))
    if frame.empty:
        raise ValueError("Tidak ada gambar di " + str(DATA_DIR))
    train_val, test = train_test_split(
        frame, test_size=0.15, stratify=frame["label"], random_state=SEED
    )
    train, val = train_test_split(
        train_val, test_size=0.1765, stratify=train_val["label"], random_state=SEED
    )
    parts = {"train": train, "val": val, "test": test}
    print("Membagi sendiri 70/15/15 secara stratified.")

names = set().union(*[set(part["label"]) for part in parts.values()])
labels = [c for c in APP_CLASSES if c in names] + sorted(names - set(APP_CLASSES))
if labels != APP_CLASSES:
    print("PERINGATAN: label", labels, "berbeda dari kelas aplikasi", APP_CLASSES)

for name, part in parts.items():
    print(f"{name:5s} {len(part):4d} gambar | {part['label'].value_counts().to_dict() if len(part) else {}}")

In [ ]:
def leaf_features(image):
    pixels = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    rows = np.arange(IMAGE_SIZE[1]) * pixels.shape[0] // IMAGE_SIZE[1]
    cols = np.arange(IMAGE_SIZE[0]) * pixels.shape[1] // IMAGE_SIZE[0]
    pixels = pixels[np.ix_(rows, cols)].astype(np.float32) / 255.0
    return np.concatenate([
        np.histogram(pixels[:, :, ch], bins=BINS, range=(0, 1), density=True)[0]
        for ch in range(3)
    ])

def features_for(frame):
    X, y, skipped = [], [], 0
    for row in frame.itertuples():
        try:
            with Image.open(row.path) as image:
                X.append(leaf_features(image))
        except (OSError, ValueError):
            skipped += 1
            continue
        y.append(labels.index(row.label))
    if skipped:
        print(f"{skipped} gambar dilewati karena tidak bisa dibaca.")
    return np.asarray(X), np.asarray(y)

X_train, y_train = features_for(parts["train"])
X_val, y_val = features_for(parts["val"])
X_test, y_test = features_for(parts["test"])
print("Train", X_train.shape, "| Val", X_val.shape, "| Test", X_test.shape)

In [ ]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=N_TREES,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
)
pipeline.fit(X_train, y_train)
print("Selesai. Jumlah pohon:", N_TREES)

In [ ]:
for name, X, y in (("val", X_val, y_val), ("test", X_test, y_test)):
    predictions = pipeline.predict(X)
    print(f"\n[{name}] accuracy = {accuracy_score(y, predictions):.4f}")
    print(classification_report(
        y, predictions, labels=range(len(labels)),
        target_names=labels, zero_division=0,
    ))

test_predictions = pipeline.predict(X_test)
cm = confusion_matrix(y_test, test_predictions, labels=range(len(labels)))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix - data uji")
plt.tight_layout()
plt.show()

In [ ]:
def export_forest(clf):
    trees = []
    for estimator in clf.estimators_:
        tree = estimator.tree_
        values = tree.value.reshape(len(tree.feature), -1)
        leaf = tree.children_left == -1
        probabilities = np.zeros_like(values)
        probabilities[leaf] = values[leaf] / values[leaf].sum(axis=1, keepdims=True)
        trees.append({
            "f": tree.feature.tolist(),
            "t": tree.threshold.tolist(),
            "l": tree.children_left.tolist(),
            "r": tree.children_right.tolist(),
            "p": [[round(float(v), 9) for v in row] for row in probabilities],
        })
    payload = {"features": BINS * 3, "labels": labels, "trees": trees}
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    payload["version"] = "banana-rf-" + digest[:12]
    return payload

def predict_forest(forest, features):
    totals = np.zeros(len(forest["labels"]))
    for tree in forest["trees"]:
        node = 0
        while tree["l"][node] != -1:
            node = tree["l"][node] if features[tree["f"][node]] <= tree["t"][node] else tree["r"][node]
        totals += tree["p"][node]
    return totals / len(forest["trees"])

forest = export_forest(pipeline.named_steps["randomforestclassifier"])

for features, expected in zip(X_test[:200], test_predictions[:200]):
    if int(np.argmax(predict_forest(forest, features))) != expected:
        raise AssertionError("Ekspor forest.json tidak cocok dengan sklearn")
assert all(abs(sum(p) - 1) < 1e-6 for tree in forest["trees"] for p in tree["p"] if any(p))
print("Ekspor cocok dengan sklearn dan probabilitas tiap daun berjumlah 1.")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata = {
    "version": forest["version"],
    "features": BINS * 3,
    "resize": list(IMAGE_SIZE),
    "interpolation": "nearest",
    "color": "RGB",
    "normalization": "outside_model",
    "pixel_range": [0, 1],
    "labels": labels,
    "n_estimators": N_TREES,
    "test_metrics": {"accuracy": float(accuracy_score(y_test, test_predictions))},
}
evaluation = {
    "model": "Random Forest",
    "metrics": {"accuracy": float(accuracy_score(y_test, test_predictions))},
    "labels": labels,
    "confusion_matrix": cm.tolist(),
}

files = {
    "forest.json": json.dumps(forest, separators=(",", ":")),
    "labels.json": json.dumps(labels, indent=2),
    "model_metadata.json": json.dumps(metadata, indent=2),
    "evaluation.json": json.dumps(evaluation, indent=2),
}
for name, content in files.items():
    (OUTPUT_DIR / name).write_text(content, encoding="utf-8")
    print("Menulis", OUTPUT_DIR / name, f"({len(content) / 1e6:.2f} MB)")

if COPY_TO_ASSETS:
    target = ROOT / "ecoplant" / "assets" / "models"
    if labels == APP_CLASSES:
        target.mkdir(parents=True, exist_ok=True)
        for name in ("forest.json", "labels.json", "model_metadata.json"):
            (target / name).write_text(files[name], encoding="utf-8")
        print("Disalin ke", target)
    else:
        print("Tidak disalin: label tidak sama dengan kelas aplikasi.")

## Selesai

Langkah berikutnya:

```powershell
cd ecoplant
flutter test
..\scripts\run-direct.ps1
```

Skrip yang sama tanpa notebook: `python training/train_random_forest.py`.
Gambar uji dengan label yang sama dari HP tetap perlu dicek manual: akurasi di atas
hanya mengukur gambar yang Anda berikan, belum kondisi lapangan.